In [1]:
from google.colab import drive
import requests
import numpy as np
import seaborn as sn
import os

drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


# Project Setup

### Identify the correct path

The previous commands failed because the directory `/content/gdrive/Shareddrives/GCPDS/databases/DEAP/` could not be found.

To help you identify the correct path, I will list the contents of `/content/gdrive/Shareddrives/`. Please examine the output and replace the incorrect path with the correct one in the subsequent cells.

In [2]:
# The 'Shareddrives' directory does not exist, as confirmed by the user.

## Unzip

In [3]:
target_dir = '/content/gdrive/MyDrive/'
if os.path.exists(target_dir):
    os.chdir(target_dir)
    print(f"Changed directory to: {os.getcwd()}")
else:
    print(f"Error: Directory not found: {target_dir}. Please ensure Google Drive is mounted and the path is correct.")

Changed directory to: /content/gdrive/MyDrive


In [4]:
if not os.path.exists('deap-dataset'):
  !unzip deap-dataset.zip
  print("deap-dataset.zip extracted.")
else:
  print("deap-dataset directory already exists. Skipping extraction.")

deap-dataset directory already exists. Skipping extraction.


The `deap-dataset.zip` archive has been extracted, creating a directory named `deap-dataset/` which contains all the dataset files and subdirectories, including `data_preprocessed_python/`.

## load sub

The videos are in the order of Experiment_id, so not in the order of presentation. This means the first video is the same for each participant.

In [5]:
target_dir = 'deap-dataset/data_preprocessed_python/'
if os.path.exists(target_dir):
    os.chdir(target_dir)
    print(f"Changed directory to: {os.getcwd()}")
else:
    print(f"Error: Directory not found: {os.path.join(os.getcwd(), target_dir)}. Please ensure 'deap-dataset.zip' was extracted correctly and contains 'data_preprocessed_python/'.")

Changed directory to: /content/gdrive/MyDrive/deapp/data_preprocessed_python


We've now navigated into the `deap-dataset/data_preprocessed_python/` directory, where the individual subject data files (`sXX.dat`) are located.

In [6]:
import pickle
import os

In [7]:
i = 1
with open('s'+ '01' + '.dat', 'rb') as file:
  subject = pickle.load(file, encoding='latin1')

In [8]:
def load_subject_data(subject_id_str):
  """Loads data for a specific subject from a .dat file."""
  file_name = f's{subject_id_str}.dat'
  with open(file_name, 'rb') as file:
    subject = pickle.load(file, encoding='latin1')
  return subject

# Example usage: Load data for subject '01'
subject_id = '01'
subject = load_subject_data(subject_id)
print(f"Successfully loaded data for subject {subject_id}.")

Successfully loaded data for subject 01.


In [9]:
subject.keys()

dict_keys(['labels', 'data'])

The `subject` dictionary contains two main keys: 'labels' and 'data'.

*   **'labels'**: This typically contains the self-assessment scores (valence, arousal, dominance, liking) for each video watched by the participant.
*   **'data'**: This holds the physiological signal data (e.g., EEG, EOG, EMG, GSR, respiration, temperature) recorded during the experiment.

In [10]:
subject['data'].shape

(40, 40, 8064)

The shape of `subject['data']` is `(40, 40, 8064)`:

*   `40`: Represents the 40 experimental trials (videos) for this participant.
*   `40`: Represents the 40 physiological channels (e.g., 32 EEG, 8 peripheral).
*   `8064`: Represents the number of data points (samples) per channel per trial. Given a sampling rate of 128 Hz and a trial duration of 63 seconds, this corresponds to `128 samples/second * 63 seconds = 8064 samples`.

In [11]:
subject['labels'].shape

(40, 4)

The shape of `subject['labels']` is `(40, 4)`:

*   `40`: Represents the 40 experimental trials (videos).
*   `4`: Represents the 4 self-assessment scores: Valence, Arousal, Dominance, and Liking, typically rated on a scale of 1-9.

# prova

# evaluate_metrics

In [12]:
import numpy as np
from sklearn.metrics import confusion_matrix

def evaluate_metrics(y_true, y_pred):

    cm = confusion_matrix(y_true, y_pred)

    if cm.shape[0] < 2:
        return np.nan, np.nan, np.nan, np.nan

    TN = cm[0,0]
    FP = cm[0,1]
    FN = cm[1,0]
    TP = cm[1,1]

    accuracy = (TP + TN) / (TP + TN + FP + FN)

    precision = TP / (TP + FP + np.finfo(float).eps)

    recall = TP / (TP + FN + np.finfo(float).eps)

    f1 = 2 * (precision * recall) / \
         (precision + recall + np.finfo(float).eps)

    return accuracy, precision, recall, f1

# main

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt
from scipy.stats import skew, kurtosis

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

In [ ]:
# =====================================================
# DEAP EMOTION RECOGNITION
# ALL 32 SUBJECTS
# =====================================================

Fs = 128
label_column = 0

# =====================================================
# FILTER
# =====================================================

b, a = butter(
    4,
    [0.5/(Fs/2), 45/(Fs/2)],
    btype='bandpass'
)

# =====================================================
# SEGMENTS
# =====================================================

segment_length = 15 * Fs

segment_names = [
    "0-15 s",
    "15-30 s",
    "30-45 s",
    "45-60 s"
]

classifier_names = [
    "SVM",
    "KNN",
    "Logistic Regression",
    "Decision Tree",
    "LDA"
]

results = []

In [ ]:

def runTrial() -> []:
    eeg = data[trial,0:32,:]

    # =========================================
    # FILTERING
    # =========================================

    eeg_filt = np.zeros_like(eeg)

    for ch in range(32):

        eeg_filt[ch,:] = filtfilt(
            b,
            a,
            eeg[ch,:]
        )

    # =========================================
    # CURRENT SEGMENT
    # =========================================

    start_idx = segment * segment_length

    end_idx = (segment + 1) * segment_length

    eeg_seg = eeg_filt[:,start_idx:end_idx]

    # =========================================
    # FEATURE EXTRACTION
    # =========================================

    feature_vector = []

    for ch in range(32):

        x = eeg_seg[ch,:]

        mean_val = np.mean(x)

        std_val = np.std(x)

        var_val = np.var(x)

        rms_val = np.sqrt(
            np.mean(x**2)
        )

        skew_val = skew(x)

        kurt_val = kurtosis(x)

        feature_vector.extend([
            mean_val,
            std_val,
            var_val,
            rms_val,
            skew_val,
            kurt_val
        ])

    return [feature_vector, current_labels[trial]]

In [ ]:

# =====================================================
# LOOP OVER SEGMENTS
# =====================================================

for segment in range(4):

    print("\n")
    print("="*40)
    print(f"SEGMENT {segment+1}")
    print("="*40)

    X = []
    Y = []

    # =================================================
    # LOAD ALL SUBJECTS
    # =================================================

    for subject in range(1,33):

        filename = f"s{subject:02d}.dat"

        print("Loading", filename)

        with open(filename, "rb") as f:
            dataset = pickle.load(
                f,
                encoding="latin1"
            )

        data = dataset["data"]
        labels = dataset["labels"]

        current_labels = labels[:, label_column]

        current_labels = np.where(
            current_labels > 5,
            1,
            0
        )

        # =============================================
        # ALL TRIALS
        # =============================================

        for trial in range(40):
            trial_data = runTrial()

            X.append(trial_data[0])
            Y.append(trial_data[1])
            

    # =================================================
    # NORMALIZATION
    # =================================================

    X = np.array(X)
    Y = np.array(Y)

    scaler = StandardScaler()

    X = scaler.fit_transform(X)


# Methods
---

In [ ]:

# =================================================
# PCA
# =================================================

pca = PCA(
    n_components=20
)

X_red = pca.fit_transform(X)

# =================================================
# TRAIN TEST SPLIT
# =================================================

Xtrain, Xtest, Ytrain, Ytest = train_test_split(
    X_red,
    Y,
    test_size=0.30,
    random_state=42,
    stratify=Y
)

In [ ]:
# =================================================
# CLASSIFIER FUNCTIONS
# =================================================

def run_svm(Xtrain, Xtest, Ytrain, Ytest, segment):
    model = SVC(kernel='rbf')
    model.fit(Xtrain, Ytrain)
    Ypred = model.predict(Xtest)
    acc, prec, rec, f1 = evaluate_metrics(Ytest, Ypred)
    return [segment+1, 1, acc, prec, rec, f1]

def run_knn(Xtrain, Xtest, Ytrain, Ytest, segment):
    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(Xtrain, Ytrain)
    Ypred = model.predict(Xtest)
    acc, prec, rec, f1 = evaluate_metrics(Ytest, Ypred)
    return [segment+1, 2, acc, prec, rec, f1]

def run_logistic_regression(Xtrain, Xtest, Ytrain, Ytest, segment):
    model = LogisticRegression(max_iter=5000)
    model.fit(Xtrain, Ytrain)
    Ypred = model.predict(Xtest)
    acc, prec, rec, f1 = evaluate_metrics(Ytest, Ypred)
    return [segment+1, 3, acc, prec, rec, f1]

def run_decision_tree(Xtrain, Xtest, Ytrain, Ytest, segment):
    model = DecisionTreeClassifier()
    model.fit(Xtrain, Ytrain)
    Ypred = model.predict(Xtest)
    acc, prec, rec, f1 = evaluate_metrics(Ytest, Ypred)
    return [segment+1, 4, acc, prec, rec, f1]

def run_lda(Xtrain, Xtest, Ytrain, Ytest, segment):
    model = LinearDiscriminantAnalysis()
    model.fit(Xtrain, Ytrain)
    Ypred = model.predict(Xtest)
    acc, prec, rec, f1 = evaluate_metrics(Ytest, Ypred)
    return [segment+1, 5, acc, prec, rec, f1]


In [ ]:

# =================================================
# RUN ALL CLASSIFIERS
# =================================================

results.append(run_svm(Xtrain, Xtest, Ytrain, Ytest, segment))
results.append(run_knn(Xtrain, Xtest, Ytrain, Ytest, segment))
results.append(run_logistic_regression(Xtrain, Xtest, Ytrain, Ytest, segment))
results.append(run_decision_tree(Xtrain, Xtest, Ytrain, Ytest, segment))
results.append(run_lda(Xtrain, Xtest, Ytrain, Ytest, segment))


# RESULTS TABLE
---

In [ ]:


result_table = pd.DataFrame(
    results,
    columns=[
        "Segment",
        "Classifier",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]
)

print(result_table)

# =====================================================
# BEST MODEL
# =====================================================

best_index = result_table["F1"].idxmax()

print("\nBEST MODEL\n")
print(result_table.loc[best_index])

# =====================================================
# F1 PLOT
# =====================================================

plt.figure(figsize=(10,5))

plt.bar(
    range(len(result_table)),
    result_table["F1"]
)

plt.title("F1 Score Comparison")

plt.xlabel("Experiments")

plt.ylabel("F1 Score")

plt.grid(True)

plt.show()